# DS605 Lab 4: Airbnb Price Prediction — Data Cleaning

This notebook downloads the raw Kaggle **New York City Airbnb Open Data** (`AB_NYC_2019.csv`), inspects it for data-quality issues, and produces a cleaned CSV (`AB_NYC_2019_cleaned.csv`) that the main EDA/modelling notebook builds on.


## 1. Download and Load the Raw Dataset

In [1]:
# Download the dataset from Kaggle (kagglehub caches it locally after the first run).
import kagglehub

path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")
print("Path to dataset files:", path)


c:\Users\rutup\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\rutup\.cache\kagglehub\datasets\dgomonov\new-york-city-airbnb-open-data\versions\3


In [2]:
# Load the raw CSV into a DataFrame. `df` is kept untouched throughout this notebook
# so we always have the original data to compare against.
import pandas as pd

df = pd.read_csv(path + "/AB_NYC_2019.csv")


## 2. Initial Inspection

In [3]:
# Preview the first 5 rows to see what the raw data actually looks like.
df.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [4]:
# Check row/column counts, dtypes, and non-null counts per column in one summary.
# 48,895 rows, 16 columns — several columns (name, host_name, last_review,
# reviews_per_month) already show fewer non-null values than the total row count.
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  str    
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  str    
 4   neighbourhood_group             48895 non-null  str    
 5   neighbourhood                   48895 non-null  str    
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  str    
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     38843 non-n

## 3. Missing Values and Column Decisions

In [5]:
# Exact missing-value counts per column.
# Result: name (16), host_name (21), last_review (10,052), reviews_per_month (10,052)
# all have missing values; every other column is fully populated.
df.isnull().sum()


id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [6]:
# last_review and reviews_per_month are both missing for the SAME ~20.6% of listings
# (listings that have never been reviewed) — confirming the two missing-value
# columns above are linked, not two separate data problems.
(df.last_review.isnull().sum() / len(df)) * 100


np.float64(20.55833929849678)

### Cleaning decisions

- **Drop `id`** — a row identifier with no predictive value.
- **Drop `host_name`** — a free-text name with no predictive value and some missing values.
- **Keep `host_id`** — not a feature on its own, but needed later to group listings by host (e.g. counting how many listings each host manages).
- **Fill missing `name` values with `'unknown'`** rather than dropping those rows, since only the name is missing and the rest of the listing's data is still usable.
- **Convert `last_review` into `days_since_last_review`** (a numeric feature: days between the most recent review in the whole dataset and each listing's own last review), then drop the original date column — raw dates aren't directly usable by most models, but the gap in days is.
- **Fill missing `days_since_last_review` with 0`** — a missing last-review date means the listing has never been reviewed, not that the value is unknown, so 0 is the correct fill (later corrected for interpretation, not left as a placeholder).
- **Drop `reviews_per_month`** — missing for the exact same listings as `last_review`, and redundant with `number_of_reviews`, so it's removed rather than imputed.


## 4. Apply the Cleaning Steps

In [7]:
# Build the cleaned DataFrame `df1`, leaving the original `df` untouched.

# Drop the two columns identified above as having no predictive value.
df1 = df.drop(['id', 'host_name'], axis=1)

# Missing listing names just mean the host left the field blank — fill rather than drop.
df1['name'] = df1['name'].fillna('unknown')

# Drop reviews_per_month: missing for the same listings as last_review, and redundant
# with number_of_reviews.
df1 = df1.drop(['reviews_per_month'], axis=1)

# Turn the raw last_review date into a numeric feature: how many days before the most
# recent review in the ENTIRE dataset was this particular listing last reviewed.
reference_date = df1["last_review"].astype('datetime64[ns]').max()
days_since_last_review = (reference_date - df1["last_review"].astype('datetime64[ns]')).dt.days
df1['days_since_last_review'] = days_since_last_review

# A missing last_review means the listing has never been reviewed, so 0 days-since-review
# is the correct fill here (not a placeholder for "unknown").
df1.fillna({col: 0 for col in df1.columns if col == 'days_since_last_review'}, inplace=True)

# The raw date column is no longer needed now that its useful information has been
# extracted into days_since_last_review.
df1 = df1.drop(['last_review'], axis=1)


In [8]:
# Sanity check: confirm exactly which columns were added vs. dropped compared to the
# raw dataset, so the cleaning steps above did what was intended and nothing else.
print("Columns added:")
for col in df1.columns:
    if col not in df.columns:
        print(col)

print("\nColumns dropped:")
for col in df.columns:
    if col not in df1.columns:
        print(col)


Columns added:
days_since_last_review

Columns dropped:
id
host_name
last_review
reviews_per_month


## 5. Verify the Cleaned Dataset

In [9]:
# Preview the cleaned data.
df1.head()


,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,calculated_host_listings_count,availability_365,days_since_last_review
0,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,6,365,262.0
1,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2,355,48.0
2,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,1,365,0.0
3,Cozy Entire Floor of Brownstone,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,1,194,3.0
4,Entire Apt: Spacious Studio/Loft by central park,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,1,0,231.0


In [10]:
# Final check: confirm there are no missing values left anywhere in the cleaned dataset.
df1.isnull().sum()


name                              0
host_id                           0
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
calculated_host_listings_count    0
availability_365                  0
days_since_last_review            0
dtype: int64

## 6. Export the Cleaned Dataset

In [11]:
# Save the cleaned dataset to CSV so the EDA/modelling notebook can load it directly
# without repeating any of the cleaning steps above.
df1.to_csv('AB_NYC_2019_cleaned.csv', index=False)
